In [ ]:
# Match_Soldiers_To_Parents_Affiliation.py
# 7/3/20252025

# SQL: ABMC_FAG_Soldiers_Baseline ==> ABMC_FAG_Mereged.csv
# fileInputSoldier=grave_source+"_FAG_Mereged"
# fileInputABMC=grave_source+"_Relative_Pre_"+letter
# fileInputCemetery="FAG_Cemetrety_Affiliation"
# fileOutput=grave_source+"_Soldier_Correlation_Affiliation_"+letter

# fileInSold=DirMain+"Analysis\\"+fileInputSoldier+".csv"
# fileInABMC=DirMain+"Analysis\\"+fileInputABMC+".csv"
# fileInCem=DirMain+"OutputData\\Cemeteries\\"+fileInputCemetery+".csv"
# fileOut=DirMain+"Analysis\\"+fileOutput+".csv"


from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
import json
import sys
import random
import numpy as np


from dateutil import parser
from dateutil.relativedelta import relativedelta
import ast

from rapidfuzz import fuzz

def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def  display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])
        input('')

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC','TEC5'\
                'PVT','S Sgt','Capt','Pfc','TSgt','COL','Tec5','TEC4','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant','TEC 5','Maj','GM3','TEC 4','Tec4','RM3','Corp','T 5','MS','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df      

def discard_empty_sets(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:x.discard('') or x if isinstance(x,set) else x)
    return dataframe_df

def  trim_dataframe_set(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:{item.strip() for item in x if isinstance(item,str)} if isinstance(x,set) else x)
    return dataframe_df

def match_sets(soldier_match_list_df,search,column_search): 
    search_set= ast.literal_eval(search)
    if isinstance(soldier_match_list_df[column_search].iloc[0], set)==False:
        soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('{') else set()) 
    soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
        lambda x: {'nan'} if pd.isna(x) else x)
    soldier_match_list_df = soldier_match_list_df.copy()
    soldier_match_list_df.loc[:,'set_difference']=search_set-soldier_match_list_df[column_search]
    return soldier_match_list_df

def discard_set_one_character(data_set_df,column):
    data_set_df=data_set_df.copy()
    data_set_df[column] = data_set_df[column].apply(
        lambda x: {item for item in ast.literal_eval(x) if len(item) > 2} if isinstance(x, str) else 
              ({item for item in x if len(item) > 1} if isinstance(x, set) else x))
    return data_set_df

def drop_sets_dataframe_duplicates(dataframe_set_df):
    set_columns = [col for col in dataframe_set_df.columns]
    # print(set_columns)
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df

def convert_dataframe_to_set(dataframe_df,column,delimiter):
    column_set=column+'_set'
    dataframe_df_split=dataframe_df[column].str.split(delimiter)
    dataframe_df[column_set]=dataframe_df_split.apply(set)
    dataframe_df=discard_empty_sets(dataframe_df,column_set)
    dataframe_df=trim_dataframe_set(dataframe_df,column_set)   
    dataframe_df=discard_set_one_character(dataframe_df,column_set)
    return dataframe_df

def clean_name_set(name_set):
    if not isinstance(name_set, set):
        return set()
    return {
        name.strip()
        for name in name_set
        if isinstance(name, str) and len(name.strip()) > 1}

def clean_name_set_column(name_set):
    if not isinstance(name_set, set):
        return set()
    cleaned_names = set()
    for name in name_set:
        if not isinstance(name, str):
            continue
        # Split name into parts, remove single letters and empty strings
        parts = [p.strip() for p in name.split() if len(p.strip()) > 1]
        if parts:
            cleaned_names.add(' '.join(parts))  # Recombine cleaned parts
    return cleaned_names

def split_set_into_sets(dataframe_set_df,column):
    dataframe_set_df=dataframe_set_df.copy()
    dataframe_set_df[column] = dataframe_set_df[column].apply(
    lambda x: {part.strip() for item in x if isinstance(item, str) for part in item.split() if part.strip()} if isinstance(x, set) else set())
    return dataframe_set_df

def dataframe_contains_match(dataframe_df,search_column,match_column):
    column_out=search_column+'_match'
    dataframe_df[column_out]=dataframe_df.apply(
        lambda row: row[match_column] in row[search_column] if isinstance(row[search_column], str) else False,axis=1)
    return dataframe_df

def  match_2_set_columns(searcher_df,searchee_df,searcher_column,searchee_column):
    # searcher_df_split=searcher_df[searcher_column].str.split(' ')
    # searcher_df=searcher_df_split.apply(set)   
    explode_searchee_df=searchee_df.explode(searchee_column)
    explode_searchee_df_split=explode_searchee_df[searchee_column].str.split(' ')
    explode_searchee_df=explode_searchee_df_split.apply(set)
    searcher_matches=searcher_df[searcher_column]-explode_searchee_df
    searcher_matches
    return searcher_matches

def match_soldier_to_child(soldier_df,child_df,soldier_set_column,children_set_column):
    match_set=match_2_set_columns(soldier_df,child_df,soldier_set_column,children_set_column)  
    child_matches_df=pd.DataFrame(match_set,columns=['child_match'])
    match_index = child_matches_df[child_matches_df['child_match']==set()].index
    
    # Initialized Column to Boolian
    soldier_df['children_match']=False
    soldier_df.loc[match_index, 'children_match'] =True
    return soldier_df

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Match Soldiers, Parents And Affiliation'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')

DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

grave_source='ABMC'
letter='A'
# grave_source='Arlington'

relative_list=['Parents','Siblings','Spouses','Children']
fileInputSoldier=grave_source+"_FAG_Mereged"
fileInputABMC=grave_source+"_Relative_Pre_"+letter
fileInputCemetery="Categorized_Cemetery_List"
fileOutput=grave_source+"_Soldier_Correlation_Affiliation_"+letter

fileInputMissed="Missed_Soldiers_List"
fileInputMiss=DirMain+"Analysis\\"+fileInputMissed+".csv"
missed_soldiers_df=pd.read_csv(fileInputMiss)

fileInSold=DirMain+"Analysis\\"+fileInputSoldier+".csv"
fileInABMC=DirMain+"Analysis\\"+fileInputABMC+".csv"
fileInCem=DirMain+"OutputData\\Cemeteries\\"+fileInputCemetery+".csv"
fileOut=DirMain+"Analysis\\"+fileOutput+".csv"

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% Read In %%%%%%%%%%%%%%
soldiers_df=pd.read_csv(fileInSold,encoding='ISO-8859-1')
print('• soldiers_df * Download Complete *')
# abmc_fag_df=pd.read_csv(fileInABMC,encoding='ISO-8859-1',low_memory=False, nrows=10000)
abmc_fag_df=pd.read_csv(fileInABMC,encoding='ISO-8859-1',low_memory=False)
print('• abmc_fag_df * Download Complete *')
cemetery_df=pd.read_csv(fileInCem,encoding='ISO-8859-1')
print('• cemetery_df * Download Complete *')
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% Read In %%%%%%%%%%%%%%


soldiers_df=soldiers_df[soldiers_df['name_index'].str[0]==letter]
na_index=soldiers_df[soldiers_df['middle_name'].isna()].index
soldiers_df.loc[na_index,'middle_name']=''
soldiers_df['soldiers_full_name2']=soldiers_df['first_name']+' '+soldiers_df['middle_name']+' '+soldiers_df['last_name']

field_list_soldiers=['name_index', 'last_name', 'first_name', 'middle_name', 'soldiers_full_name', 'birthDate',\
                     'birth_index', 'deathDate', 'death_index', 'entry_state', 'birthCityName', 'birthStateName',\
                     'birthCountryName', 'links', 'Parents', 'Spouses', 'Siblings', 'Children']

field_list_fag=['name_index', 'name_index_short', 'last_name', 'first_name', 'middle_name', 'full_name', 'birth_date',\
                'death_date', 'birth_index', 'death_index', 'birth_city', 'birth_state', 'birth_country', 'cemetery_name',\
                'cemetery_id', 'url', 'spouses', 'siblings', 'children']

field_list_cemetery=['cemetery_id', 'memorial_count', 'cemetery', 'city', 'county', 'state', 'county_id', 'state_id', 'prob_jewish', 'low_count_flag']

soldiers_df['birth_index'] = pd.to_numeric(soldiers_df['birth_index'].fillna(20001010),errors='coerce').astype('Int64')
soldiers_df['death_index'] = pd.to_numeric(soldiers_df['death_index'].fillna(20001010),errors='coerce').astype('Int64')

abmc_fag_df['birth_index'] = pd.to_numeric(abmc_fag_df['birth_index'].fillna(20001010),errors='coerce').astype('Int64')
abmc_fag_df['death_index'] = pd.to_numeric(abmc_fag_df['death_index'].fillna(20001010),errors='coerce').astype('Int64')
abmc_fag_df['birth_date']=pd.to_datetime(abmc_fag_df['birth_date'].fillna('01/01/2000'),format='mixed',errors='coerce')


field_list=['name_index', 'last_name', 'first_name', 'middle_name','soldiers_full_name', 'birthDate', 'birth_index', 'deathDate',\
            'death_index','entry_state', 'birthCityName', 'birthStateName', 'birthCountryName', 'links', 'Parents', 'Spouses', 'Siblings', 'Children']
field_list_set=set(field_list)

# $$$$$$$$$$$ Clean databases fields $$$$$$$$$$$$$$$$$$$$$
column_set=['soldiers_full_name','Parents', 'Spouses', 'Siblings', 'Children']
for column in column_set:
    soldiers_df=clean_dataframe(soldiers_df,column)

column_set=['full_name', 'spouses', 'siblings', 'children']
for column in column_set:
    abmc_fag_df=clean_dataframe(abmc_fag_df,column)
print('* Clean Databases Fields Complete *')
# $$$$$$$$$$$$$ Convert dataframe fields to sets $$$$$$$$$$$$$$$$$$$$$$$
star='*'
space=' '

soldiers_df_column_list=['soldiers_full_name2']
for column in soldiers_df_column_list:
    soldiers_df=convert_dataframe_to_set(soldiers_df,column,space)
    soldiers_df=clean_dataframe(soldiers_df,column)
    
soldiers_df_set_column_list= ['Parents', 'Spouses', 'Siblings']
for column in soldiers_df_set_column_list:
    soldiers_df=convert_dataframe_to_set(soldiers_df,column,star)
    column_set=column+'_set'
    soldiers_df[column_set] = soldiers_df[column_set].apply(clean_name_set_column)

print('* Soldier Fields Convert to Sets • Complete *')

fag_df_column_list=['full_name']
for column in fag_df_column_list:
    abmc_fag_df=convert_dataframe_to_set(abmc_fag_df,column,space)
    abmc_fag_df=clean_dataframe(abmc_fag_df,column)
    
abmc_fag_df_set_column_list= ['spouses', 'siblings', 'children']
for column in abmc_fag_df_set_column_list:
    abmc_fag_df=convert_dataframe_to_set(abmc_fag_df,column,star)
    column_set=column+'_set'
    abmc_fag_df[column_set] = abmc_fag_df[column_set].apply(clean_name_set_column)

print('* abmc_fag Fields Convert to Sets • Complete *')

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())

FAG_Mereged_length=len(soldiers_df)
Relative_Pre_length=len(abmc_fag_df)
Cemetery_list_length=len(cemetery_df)

print(f"** Downloading Complete **** • FAG_Mereged Length: {FAG_Mereged_length} •  Relative_Pre Length: {Relative_Pre_length} •  Cemetery_list Length: {Cemetery_list_length} ••  Processing Time: {processing_time_seconds} Sec")
input('')

#$$$$$$$$$$$$$$$$$$$$$ Merge each soldier $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
maximize_screen()
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)
soldier_line_unknown=[soldiers_full_name,soldiers_index,parent,spouse,'Unknown','Unknown','Unknown','Unknown','Unknown','Unknown']
soldier_match_df=pd.DataFrame([])
matched_count=0
for index1, row_soldier in soldiers_df.iterrows():
    soldiers_index=row_soldier['name_index']
    soldiers_full_name=row_soldier['soldiers_full_name2']
    soldiers_name_set=row_soldier['soldiers_full_name2_set']
    parents_set=row_soldier['Parents_set']
    last_name=row_soldier['last_name']
    abmc_fag_last_name_df=abmc_fag_df[abmc_fag_df['last_name']==last_name]
    parents_list=list(parents_set)
    if parents_list==['nan']:
        parent='Unknown'
        spouse='Unknown'
        continue
    elif len( parents_list)==1:
        parent=parents_list[0]
        spouse='Unknown'
    else:
        parent=parents_list[0]
        spouse=parents_list[1]
    for parent_item in parents_list:  
        parents_split=parent_item.split(' ')
        parent_test_set=set(parents_split) 
        spouse_split=parent_item.split(' ')
        spouse_test_set=set(spouse_split) 
        missing_name=parent_test_set-abmc_fag_last_name_df['full_name_set']
        matched_parent_df=abmc_fag_last_name_df[missing_name==set()]
        if len(matched_parent_df)>1: 
            missing_name=spouse_test_set-matched_parent_df['spouses_set']
            matched_parent_df=matched_parent_df[missing_name==set()]
        if len(matched_parent_df)==0:continue
        matched_spouse_name=matched_parent_df['spouses'].iloc[0]
        matched_parent_name=matched_parent_df['full_name']
        matched_cemetery_name=matched_parent_df['cemetery_name'].iloc[0]
        matched_cemetery_id=matched_parent_df['cemetery_id'].iloc[0]
        if matched_cemetery_id!='' and (cemetery_df['cemetery_id']==matched_cemetery_id).sum()>0:
            probability_jewish=round(100*cemetery_df['prob_jewish'][cemetery_df['cemetery_id']==matched_cemetery_id].iloc[0],1)
            low_confidence_flag=cemetery_df['low_count_flag'][cemetery_df['cemetery_id']==matched_cemetery_id].iloc[0]
        else:
            cemetery_affiliation='Unknown'
            low_confidence_flag='Unknown'
        matched_children_set=matched_parent_df['children_set']
        matched_parent_explode_df=matched_parent_df.explode('children_set')
        matched_parent_split=matched_parent_explode_df['children_set'].str.split(' ')
        children_name_set=matched_parent_split.apply(set)
        name_missing_set=soldiers_name_set-children_name_set
        matched_child_df=matched_parent_explode_df[name_missing_set==set()]
        matched_child_name=soldiers_full_name
        if parents_list==['nan'] or len(name_missing_set)==0:
            soldier_line=soldier_line_unknown
        else:
            soldier_line=[soldiers_full_name,soldiers_index,parent,spouse,matched_spouse_name,matched_child_name,probability_jewish,low_confidence_flag,matched_cemetery_name,matched_cemetery_id]
        soldier_line_df=pd.DataFrame(soldier_line).T
        soldier_line_df.columns=['soldiers_name','soldiers_index','parent_name','spouse_name','matched_spouse_name','matched_child_name','probability_jewish','low_confidence_flag','matched_cemetery_name',\
                                 'matched_cemetery_id']       
        soldier_match_df=pd.concat([soldier_match_df,soldier_line_df])
                               
    # if soldier_line_df['probability_jewish'].iloc[0]!='Unknown':matched_count+=1;print(f" ** Match Count: {matched_count} • soldier: {soldier_line_df['soldiers_name']}  • Index: {soldier_line_df['soldiers_index']}  • parent: {soldier_line_df['parent_name']}  • spouse: {soldier_line_df['spouse_name']}  • matched_spouse: {soldier_line_df['matched_spouse_name']}  • child: {soldier_line_df['matched_child_name']}  • probability_jewish: {soldier_line_df['probability_jewish']} • low_confidence: {soldier_line_df['low_confidence_flag']}  • cemetery_name: {soldier_line_df['matched_cemetery_name']}")
soldier_match_df = soldier_match_df.drop_duplicates()        
soldier_match_df.reset_index(drop=True, inplace = True)
soldier_match_df.to_csv(fileOut,mode="w",index=False,header=True)
print("\n")
print('UploadedTo:',fileOut,'Number of rows:',len(soldier_match_df))  
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")     

In [ ]:
#$$$$$$$$$$$$$$$$$$$$$ Merge each soldier $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
maximize_screen()
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)
soldier_line_unknown=[soldiers_full_name,soldiers_index,parent,spouse,'Unknown','Unknown','Unknown','Unknown','Unknown','Unknown']
soldier_match_df=pd.DataFrame([])
matched_count=0
for index1, row_soldier in soldiers_df.iterrows():
    soldiers_index=row_soldier['name_index']
    
    if soldiers_index!='Adams-Howard-19421126':continue 
    input('')
       
    soldiers_full_name=row_soldier['soldiers_full_name2']
    soldiers_name_set=row_soldier['soldiers_full_name2_set']
    parents_set=row_soldier['Parents_set']
    last_name=row_soldier['last_name']
    abmc_fag_last_name_df=abmc_fag_df[abmc_fag_df['last_name']==last_name]
    parents_list=list(parents_set)
    if parents_list==['nan']:
        parent='Unknown'
        spouse='Unknown'
        continue
    elif len( parents_list)==1:
        parent=parents_list[0]
        spouse='Unknown'
    else:
        parent=parents_list[0]
        spouse=parents_list[1]
    for parent_item in parents_list:  
        parents_split=parent_item.split(' ')
        parent_test_set=set(parents_split) 
        spouse_split=parent_item.split(' ')
        spouse_test_set=set(spouse_split) 
        missing_name=parent_test_set-abmc_fag_last_name_df['full_name_set']
        matched_parent_df=abmc_fag_last_name_df[missing_name==set()]
        if len(matched_parent_df)>1: 
            missing_name=spouse_test_set-matched_parent_df['spouses_set']
            matched_parent_df=matched_parent_df[missing_name==set()]
        if len(matched_parent_df)==0:continue
        matched_spouse_name=matched_parent_df['spouses'].iloc[0]
        matched_parent_name=matched_parent_df['full_name']
        matched_cemetery_name=matched_parent_df['cemetery_name'].iloc[0]
        matched_cemetery_id=matched_parent_df['cemetery_id'].iloc[0]
        if matched_cemetery_id!='' and (cemetery_df['cemetery_id']==matched_cemetery_id).sum()>0:
            probability_jewish=round(100*cemetery_df['prob_jewish'][cemetery_df['cemetery_id']==matched_cemetery_id].iloc[0],1)
            low_confidence_flag=cemetery_df['low_count_flag'][cemetery_df['cemetery_id']==matched_cemetery_id].iloc[0]
        else:
            cemetery_affiliation='Unknown'
            low_confidence_flag='Unknown'
        matched_children_set=matched_parent_df['children_set']
        matched_parent_explode_df=matched_parent_df.explode('children_set')
        matched_parent_split=matched_parent_explode_df['children_set'].str.split(' ')
        children_name_set=matched_parent_split.apply(set)
        name_missing_set=soldiers_name_set-children_name_set
        matched_child_df=matched_parent_explode_df[name_missing_set==set()]
        matched_child_name=soldiers_full_name
        if parents_list==['nan'] or len(name_missing_set)==0:
            soldier_line=soldier_line_unknown
        else:
            soldier_line=[soldiers_full_name,soldiers_index,parent,spouse,matched_spouse_name,matched_child_name,probability_jewish,low_confidence_flag,matched_cemetery_name,matched_cemetery_id]
        soldier_line_df=pd.DataFrame(soldier_line).T
        soldier_line_df.columns=['soldiers_name','soldiers_index','parent_name','spouse_name','matched_spouse_name','matched_child_name','probability_jewish','low_confidence_flag','matched_cemetery_name',\
                                 'matched_cemetery_id']
        
        soldier_match_df=pd.concat([soldier_match_df,soldier_line_df])
                               
    # if soldier_line_df['probability_jewish'].iloc[0]!='Unknown':matched_count+=1;print(f" ** Match Count: {matched_count} • soldier: {soldier_line_df['soldiers_name']}  • Index: {soldier_line_df['soldiers_index']}  • parent: {soldier_line_df['parent_name']}  • spouse: {soldier_line_df['spouse_name']}  • matched_spouse: {soldier_line_df['matched_spouse_name']}  • child: {soldier_line_df['matched_child_name']}  • probability_jewish: {soldier_line_df['probability_jewish']} • low_confidence: {soldier_line_df['low_confidence_flag']}  • cemetery_name: {soldier_line_df['matched_cemetery_name']}")
soldier_match_df = soldier_match_df.drop_duplicates()        
soldier_match_df.reset_index(drop=True, inplace = True)
soldier_match_df.to_csv(fileOut,mode="w",index=False,header=True)
print("\n")
print('UploadedTo:',fileOut,'Number of rows:',len(soldier_match_df))  
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   

In [ ]:
fileInputMissed="Missed_Soldiers_List"
fileInputMiss=DirMain+"Analysis\\"+fileInputMissed+".csv"
missed_soldiers_df=pd.read_csv(fileInputMiss)

for index0,row_missed in missed_soldiers_df.iterrows():
    select_soldiers_index=row_missed['missed_soldiers']
    
    # print('select_soldiers_index:',select_soldiers_index)
    # input('')
    for index1, row_soldier in soldiers_df.iterrows():
        soldiers_index=row_soldier['name_index']
            
        if soldiers_index!=select_soldiers_index:continue 
        soldiers_full_name=row_soldier['soldiers_full_name2']
        soldiers_name_set=row_soldier['soldiers_full_name2_set']
        parents_set=row_soldier['Parents_set']
        last_name=row_soldier['last_name']
        abmc_fag_last_name_df=abmc_fag_df[abmc_fag_df['last_name']==last_name]
        parents_list=list(parents_set)
        print(select_soldiers_index,'parents_list:',parents_list)
        if parents_list==['nan']:
            parent='Unknown'
            spouse='Unknown'
        elif len( parents_list)==1:
            parent=parents_list[0]
            spouse='Unknown'
        else:
            parent=parents_list[0]
            spouse=parents_list[1]
        soldier_line_unknown=[soldiers_full_name,soldiers_index,parent,spouse,'Unknown','Unknown','Unknown','Unknown','Unknown','Unknown'] 
        for parent_item in parents_list:  
            parents_split=parent_item.split(' ')
            parent_test_set=set(parents_split) 
            spouse_split=spouse.split(' ')
            spouse_test_set=set(spouse_split) 
            missing_parent_df=parent_test_set-abmc_fag_last_name_df['full_name_set']       
            matched_parent_df=abmc_fag_last_name_df[missing_parent_df==set()]          
            if len(matched_parent_df)>1:
                if len(parents_list)==2:
                    abmc_fag_last_name_df=split_set_into_sets(abmc_fag_last_name_df,'spouses_set')
                    missing_spouse_df=spouse_test_set-abmc_fag_last_name_df['full_name_set']  
                    matched_spouse_df=abmc_fag_last_name_df[missing_spouse_df==set()]
                    if len(matched_spouse_df)==0:
                        soldier_line=soldier_line_unknown
                        soldier_line_df=pd.DataFrame(soldier_line).T
                        soldier_line_df.columns=['soldiers_name','soldiers_index','parent_name','spouse_name','matched_spouse_name','matched_child_name','probability_jewish','low_confidence_flag','matched_cemetery_name',\
                                             'matched_cemetery_id']           
                        soldier_match_df=pd.concat([soldier_match_df,soldier_line_df])
                        continue
                        # print(soldier_line_df)
                    else:
                        column_list=list(matched_parent_df)                  
                        matched_parent_spouse_df=pd.merge(matched_parent_df,matched_spouse_df,left_on=['spouses','full_name'],right_on=['full_name','spouses'],how='inner')
                        column_list_merged=[col + '_x' for col in list(matched_parent_df)]
                        matched_parent_df[column_list]=matched_parent_spouse_df[column_list_merged]
                elif len(parents_list)==1:
                    matched_parent_df=matched_parent_df[matched_parent_df['children'].str.contains(soldiers_full_name)]   
            else:
                soldier_line=soldier_line_unknown
                soldier_line_df=pd.DataFrame(soldier_line).T
                soldier_line_df.columns=['soldiers_name','soldiers_index','parent_name','spouse_name','matched_spouse_name','matched_child_name','probability_jewish','low_confidence_flag','matched_cemetery_name',\
                                     'matched_cemetery_id']           
                soldier_match_df=pd.concat([soldier_match_df,soldier_line_df])
                # print(soldier_line_df)
                # input('')
                continue
            print('\n*soldiers_index*:',soldiers_index,'\nlen matched_parent_df=',len(matched_parent_df),'\n')
            input('')
            matched_spouse_name=matched_parent_df['spouses'].iloc[0]
            matched_parent_name=matched_parent_df['full_name']
            matched_cemetery_name=matched_parent_df['cemetery_name'].iloc[0]
            matched_cemetery_id=matched_parent_df['cemetery_id'].iloc[0]
            if matched_cemetery_id!='' and (cemetery_df['cemetery_id']==matched_cemetery_id).sum()>0:
                probability_jewish=round(100*cemetery_df['prob_jewish'][cemetery_df['cemetery_id']==matched_cemetery_id].iloc[0],1)
                low_confidence_flag=cemetery_df['low_count_flag'][cemetery_df['cemetery_id']==matched_cemetery_id].iloc[0]
            else:
                cemetery_affiliation='Unknown'
                low_confidence_flag='Unknown'
            matched_children_set=matched_parent_df['children_set']
            if matched_parent_df['children_set'].isna().all()==True:
                matched_child_name='None'
            else:
                matched_parent_explode_df=matched_parent_df.explode('children_set')
                matched_parent_split=matched_parent_explode_df['children_set'].str.split(' ')
                children_name_set=matched_parent_split.apply(set)
                name_missing_set=soldiers_name_set-children_name_set
                matched_child_df=matched_parent_explode_df[name_missing_set==set()]
                matched_child_name=soldiers_full_name
            if parents_list==['nan'] or len(name_missing_set)==0:
                soldier_line=soldier_line_unknown
            else:
                soldier_line=[soldiers_full_name,soldiers_index,parent,spouse,matched_spouse_name,matched_child_name,probability_jewish,low_confidence_flag,matched_cemetery_name,matched_cemetery_id]
            soldier_line_df=pd.DataFrame(soldier_line).T
            soldier_line_df.columns=['soldiers_name','soldiers_index','parent_name','spouse_name','matched_spouse_name','matched_child_name','probability_jewish','low_confidence_flag','matched_cemetery_name',\
                                     'matched_cemetery_id']
            
            soldier_match_df=pd.concat([soldier_match_df,soldier_line_df])
            # print(soldier_line_df)
            # input('')
            
            # missing_name_len_df = missing_name_df.apply(len)
            # missing_name_len_min=missing_name_len_df.min()
            # match_min_name=abmc_fag_last_name_df['full_name_set'][missing_name_len_df==missing_name_len_min]  
        # print('parents_list:',parents_list)
        # input('')    
        break
    
        # print(abmc_fag_last_name_df['full_name_set'])
        # print(matched_parent_df)
        # missing_name_len = missing_name_df.apply(len)
        # missing_name_df[missing_name_df==missing_name_min]
        # input('')
        # print(missing_name_min_index)

In [ ]:
# ************ Double_Parent_Spouse_Parent_Child_Match ****
soldiers_df['Parent_1']=soldiers_df['Parents'].str.split('*').str[0].str.strip()
soldiers_df['Parent_2']=soldiers_df['Parents'].str.split('*').str[1].str.strip()
soldiers_df1=soldiers_df[['name_index','soldiers_full_name2','soldiers_full_name2_set','Parent_1','Parent_2']]
abmc_fag_df1=abmc_fag_df[['full_name','spouses','children','children_set','cemetery_name','cemetery_id']]

matched_parent_spouse_df11=pd.merge(
    soldiers_df1,
    abmc_fag_df1,
    left_on=['Parent_1','Parent_2'],
    right_on=['full_name','spouses'],
    # left_on=['Parent_1'],
    # right_on=['full_name'],    
    how='inner')
print('matched_parent_spouse_df11-Size:',len(matched_parent_spouse_df11))

matched_parent_spouse_df211=pd.merge(
    matched_parent_spouse_df11,
    abmc_fag_df1,
    left_on=['spouses','full_name'],
    right_on=['full_name','spouses'],
    how='inner')
print('matched_parent_spouse_df211-Size:',len(matched_parent_spouse_df211))

matched_parent_spouse_df211=drop_sets_dataframe_duplicates(matched_parent_spouse_df211)
matched_parent_spouse_df211.reset_index(drop=True, inplace = True)

matched_parent_spouse_df211c=match_soldier_to_child(matched_parent_spouse_df211,matched_parent_spouse_df211,'soldiers_full_name2_set','children_set_x')

print('matched_parent_spouse_df211c-Size:',len(matched_parent_spouse_df211c))

matched_out_df211c=pd.merge(
    soldiers_df1,
    matched_parent_spouse_df211c,
    left_on=[ 'name_index'],
    right_on=['name_index'],
    how='inner')
matched_out_df211c=drop_sets_dataframe_duplicates(matched_out_df211c)
print('matched_out_df211c-Size:',len(matched_out_df211c))

matched_out_df211c=drop_sets_dataframe_duplicates(matched_out_df211c)
matched_out_df211c.reset_index(drop=True, inplace = True)
print('matched_out_df12c-Size:',len(matched_out_df211c))

reorder_format=['name_index','soldiers_full_name2_x','Parent_1_x','Parent_2_x','spouses_x','full_name_y',\
 'spouses_y','children_match','children_x','children_y','cemetery_name_x',\
 'cemetery_name_y','cemetery_id_x','cemetery_id_y','soldiers_full_name2_set_x']

matched_out_df211c=matched_out_df211c[reorder_format]
fileOutTest=DirMain+"Analysis\\Double_Parent_Spouse_Parent_Child_Match.csv"
matched_out_df211c.to_csv(fileOutTest,mode="w",index=False,header=True)
matched_out_df211c


In [308]:
list(matched_out_df12c)

['name_index',
 'soldiers_full_name2_x',
 'Parent_2_x',
 'spouses',
 'full_name',
 'children_match',
 'children',
 'cemetery_name',
 'cemetery_id']

In [ ]:
# **** Double_Parent_Child_Match *******
soldiers_df['Parent_1']=soldiers_df['Parents'].str.split('*').str[0].str.strip()
soldiers_df['Parent_2']=soldiers_df['Parents'].str.split('*').str[1].str.strip()
soldiers_df1=soldiers_df[['name_index','soldiers_full_name2','soldiers_full_name2_set','Parent_1','Parent_2']]
abmc_fag_df1=abmc_fag_df[['full_name','spouses','children','children_set','cemetery_name','cemetery_id']]

matched_parent_spouse_df12=pd.merge(
    soldiers_df1,
    abmc_fag_df1,
    left_on=['Parent_1','Parent_2'],
    right_on=['full_name','spouses'],
    # left_on=['Parent_1'],
    # right_on=['full_name'],    
    how='inner')
print('matched_parent_spouse_df12-Size:',len(matched_parent_spouse_df12))
matched_parent_spouse_df12=drop_sets_dataframe_duplicates(matched_parent_spouse_df12)
matched_parent_spouse_df12.reset_index(drop=True, inplace = True)

matched_parent_spouse_df12c=match_soldier_to_child(matched_parent_spouse_df12,matched_parent_spouse_df12,'soldiers_full_name2_set','children_set')

matched_out_df12c=pd.merge(
    soldiers_df1,
    matched_parent_spouse_df12c,
    left_on=[ 'name_index'],
    right_on=['name_index'],
    how='inner')
print('matched_out_df12c-Size:',len(matched_out_df12c))
matched_out_df12c=drop_sets_dataframe_duplicates(matched_out_df12c)
matched_out_df12c.reset_index(drop=True, inplace = True)
print('matched_out_df12c-Size:',len(matched_out_df12c))

matched_out_df12c=drop_sets_dataframe_duplicates(matched_out_df12c)
matched_out_df12c.reset_index(drop=True, inplace = True)
print('matched_out_df12c-Size:',len(matched_out_df12c))

reorder_format=['name_index','soldiers_full_name2_x','Parent_1_x','Parent_2_x','spouses','full_name',\
 'children_match','children','cemetery_name','cemetery_id']

matched_out_df12c=matched_out_df12c[reorder_format]

fileOutTest=DirMain+"Analysis\\Double_Parent_Child_Match.csv"
matched_out_df12c.to_csv(fileOutTest,mode="w",index=False,header=True)
matched_out_df12c


In [ ]:
# **** Single_Parent_Child_Match Parent 1 *******
soldiers_df['Parent_1']=soldiers_df['Parents'].str.split('*').str[0].str.strip()
soldiers_df['Parent_2']=soldiers_df['Parents'].str.split('*').str[1].str.strip()
soldiers_df1=soldiers_df[['name_index','soldiers_full_name2','soldiers_full_name2_set','Parent_1','Parent_2']]
abmc_fag_df1=abmc_fag_df[['full_name','spouses','children','children_set','cemetery_name','cemetery_id']]

matched_parent_spouse_df12=pd.merge(
    soldiers_df1,
    abmc_fag_df1,
    # left_on=['Parent_1','Parent_2'],
    # right_on=['full_name','spouses'],
    left_on=['Parent_1'],
    right_on=['full_name'],    
    how='inner')
print('matched_parent_spouse_df12-Size:',len(matched_parent_spouse_df12))
matched_parent_spouse_df12=drop_sets_dataframe_duplicates(matched_parent_spouse_df12)
matched_parent_spouse_df12.reset_index(drop=True, inplace = True)

matched_parent_spouse_df12c=match_soldier_to_child(matched_parent_spouse_df12,matched_parent_spouse_df12,'soldiers_full_name2_set','children_set')

matched_out_df12c=pd.merge(
    soldiers_df1,
    matched_parent_spouse_df12c,
    left_on=[ 'name_index'],
    right_on=['name_index'],
    how='inner')
print('matched_out_df12c-Size:',len(matched_out_df12c))
matched_out_df12c=drop_sets_dataframe_duplicates(matched_out_df12c)
matched_out_df12c.reset_index(drop=True, inplace = True)
print('matched_out_df12c-Size:',len(matched_out_df12c))

matched_out_df12c=drop_sets_dataframe_duplicates(matched_out_df12c)
matched_out_df12c.reset_index(drop=True, inplace = True)
print('matched_out_df12c-Size:',len(matched_out_df12c))

reorder_format=['name_index','soldiers_full_name2_x','Parent_1_x','spouses','full_name',\
 'children_match','children','cemetery_name','cemetery_id']

matched_out_df12c=matched_out_df12c[reorder_format]

fileOutTest=DirMain+"Analysis\\Single_Parent_Child_Match_1.csv"
matched_out_df12c.to_csv(fileOutTest,mode="w",index=False,header=True)
matched_out_df12c


In [ ]:
# **** Single_Parent_Child_Match Parent 2 *******
soldiers_df['Parent_1']=soldiers_df['Parents'].str.split('*').str[0].str.strip()
soldiers_df['Parent_2']=soldiers_df['Parents'].str.split('*').str[1].str.strip()
soldiers_df1=soldiers_df[['name_index','soldiers_full_name2','soldiers_full_name2_set','Parent_1','Parent_2']]
abmc_fag_df1=abmc_fag_df[['full_name','spouses','children','children_set','cemetery_name','cemetery_id']]

matched_parent_spouse_df12=pd.merge(
    soldiers_df1,
    abmc_fag_df1,
    # left_on=['Parent_1','Parent_2'],
    # right_on=['full_name','spouses'],
    left_on=['Parent_2'],
    right_on=['full_name'],    
    how='inner')
print('matched_parent_spouse_df12-Size:',len(matched_parent_spouse_df12))
matched_parent_spouse_df12=drop_sets_dataframe_duplicates(matched_parent_spouse_df12)
matched_parent_spouse_df12.reset_index(drop=True, inplace = True)

matched_parent_spouse_df12c=match_soldier_to_child(matched_parent_spouse_df12,matched_parent_spouse_df12,'soldiers_full_name2_set','children_set')

matched_out_df12c=pd.merge(
    soldiers_df1,
    matched_parent_spouse_df12c,
    left_on=[ 'name_index'],
    right_on=['name_index'],
    how='inner')
print('matched_out_df12c-Size:',len(matched_out_df12c))
matched_out_df12c=drop_sets_dataframe_duplicates(matched_out_df12c)
matched_out_df12c.reset_index(drop=True, inplace = True)
print('matched_out_df12c-Size:',len(matched_out_df12c))

matched_out_df12c=drop_sets_dataframe_duplicates(matched_out_df12c)
matched_out_df12c.reset_index(drop=True, inplace = True)
print('matched_out_df12c-Size:',len(matched_out_df12c))

reorder_format=['name_index','soldiers_full_name2_x','Parent_2_x','spouses','full_name',\
 'children_match','children','cemetery_name','cemetery_id']

matched_out_df12c=matched_out_df12c[reorder_format]

fileOutTest=DirMain+"Analysis\\Single_Parent_Child_Match_2.csv"
matched_out_df12c.to_csv(fileOutTest,mode="w",index=False,header=True)
matched_out_df12c
